In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('.')))

from cohort_retention.cohort import build_cohort_matrix

In [ ]:
# Constants and config
DATA_DIR = Path('..') / 'data' / 'synthetic'
LOOKBACK_DAYS = 90
COHORT_PERIOD = 'W'   # 'W' = weekly, 'M' = monthly
MAX_PERIODS = 16

In [ ]:
# Load data
events = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['occurred_at'])
print(f"{len(events):,} events  |  {events['user_id'].nunique():,} unique users")

In [ ]:
# Build cohort retention matrix
matrix = build_cohort_matrix(
    events,
    period=COHORT_PERIOD,
    max_periods=MAX_PERIODS,
)
print(f"Matrix: {matrix.shape[0]} cohorts × {matrix.shape[1]} periods")
matrix.head()

In [ ]:
# Plot retention heatmap
fig, ax = plt.subplots(figsize=(14, max(6, len(matrix) * 0.35)))

# Drop periods with no data (trailing NaN columns)
plot_data = matrix.dropna(axis=1, how='all').fillna(0)

sns.heatmap(
    plot_data,
    ax=ax,
    cmap='YlGn',
    vmin=0, vmax=1,
    annot=True,
    fmt='.0%',
    linewidths=0.5,
    cbar_kws={'label': 'Retention Rate', 'format': mticker.PercentFormatter(xmax=1)},
)

ax.set_xlabel('Weeks since first activity')
ax.set_ylabel('Cohort (week of first activity)')
ax.set_title('Weekly Cohort Retention Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Average retention curve across all cohorts
avg_retention = matrix.mean(axis=0).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, marker='o', linewidth=2)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_xlabel('Weeks since first activity')
ax.set_ylabel('Average retention rate')
ax.set_title('Average Retention Curve (all cohorts)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()